In [1]:
from pathlib import Path
import json, random

import cv2
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path("/homes/mxqasim/Chocolathon")
BOX_MANIFEST = ROOT / "data/boxes/manifest.csv"
V4_MANIFEST = ROOT / "data/synthetic_filled_boxes/v4/manifest.csv"
OLD_MODEL_DIR = ROOT / "data/experiments/box_localization_v1/outputs"
RUN_DIR = ROOT / "data/experiments/box_localization_v2"

RUN_DIR.mkdir(parents=True, exist_ok=True)

CAPACITIES = [6, 16, 30, 50]
CAPACITY_TO_INDEX = {capacity: index for index, capacity in enumerate(CAPACITIES)}
CORNER_NAMES = ("top_left", "top_right", "bottom_right", "bottom_left")

# One difficult source per capacity is reserved for validation
VALIDATION_SOURCES = {
    6: "IMG_2021.jpg",
    16: "IMG_2026.jpg",
    30: "IMG_2030.jpg",
    50: "IMG_2034.jpg"
}

boxes = pd.read_csv(BOX_MANIFEST)
synthetic = pd.read_csv(V4_MANIFEST)

records = []

# Original empty-box images
for box in boxes.itertuples():
    image_path = ROOT / "data/boxes/raw" / box.file
    annotation_path = Path(box.annotation_file)

    if not annotation_path.is_absolute():
        annotation_path = ROOT / annotation_path

    annotation = json.loads(annotation_path.read_text())
    corners = [
        annotation["corners"][name]
        for name in CORNER_NAMES
    ]

    records.append({
        "image": str(image_path),
        "source_file": box.file,
        "box_size": int(box.box_size),
        "kind": "original_empty",
        "corners": json.dumps(corners)
    })

# Corrected completely filled synthetic images
for item in synthetic.itertuples():
    records.append({
        "image": str(Path(item.image)),
        "source_file": item.source_file,
        "box_size": int(item.box_size),
        "kind": "synthetic_filled",
        "corners": item.corners
    })

dataset = pd.DataFrame(records)

dataset["split"] = dataset.apply(
    lambda row: (
        "validation"
        if row["source_file"] == VALIDATION_SOURCES[int(row["box_size"])]
        else "train"
    ),
    axis=1
)

dataset["class_index"] = dataset["box_size"].map(CAPACITY_TO_INDEX)

assert len(dataset) == 64
assert dataset["image"].map(lambda p: Path(p).exists()).all()
assert dataset["class_index"].notna().all()
assert set(dataset["split"]) == {"train", "validation"}

train_sources = set(
    dataset.loc[dataset["split"] == "train", "source_file"]
)
validation_sources = set(
    dataset.loc[dataset["split"] == "validation", "source_file"]
)

assert train_sources.isdisjoint(validation_sources)

# Each physical source contributes one original plus three v4 renders
source_counts = dataset.groupby("source_file").size()
assert source_counts.eq(4).all(), source_counts[source_counts.ne(4)]

dataset.to_csv(
    RUN_DIR / "localization_dataset.csv",
    index=False
)

print("Total images:", len(dataset))
print("Training images:", (dataset["split"] == "train").sum())
print("Validation images:", (dataset["split"] == "validation").sum())

print("\nValidation sources:")
for capacity, source in VALIDATION_SOURCES.items():
    print(f"{capacity:>2} slots: {source}")

print("\nImages by split and capacity:")
display(
    dataset.groupby(
        ["split", "box_size", "kind"]
    ).size().rename("images").to_frame()
)

Total images: 64
Training images: 48
Validation images: 16

Validation sources:
 6 slots: IMG_2021.jpg
16 slots: IMG_2026.jpg
30 slots: IMG_2030.jpg
50 slots: IMG_2034.jpg

Images by split and capacity:


images
split      box_size kind                    
train      6        original_empty         2
                    synthetic_filled       6
           16       original_empty         4
                    synthetic_filled      12
           30       original_empty         3
                    synthetic_filled       9
           50       original_empty         3
                    synthetic_filled       9
validation 6        original_empty         1
                    synthetic_filled       3
           16       original_empty         1
                    synthetic_filled       3
           30       original_empty         1
                    synthetic_filled       3
           50       original_empty         1
                    synthetic_filled       3

In [2]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b0

IMAGE_SIZE = 224
BATCH_SIZE = 8
NUM_WORKERS = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

torch.manual_seed(SEED)
torch.set_num_threads(4)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


class LocalizationDataset(Dataset):
    def __init__(self, frame, augment=False):
        self.frame = frame.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        bgr = cv2.imread(row["image"])

        if bgr is None:
            raise ValueError(f"Cannot decode image: {row['image']}")

        image = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        height, width = image.shape[:2]

        corners = np.asarray(
            json.loads(row["corners"]),
            dtype=np.float32
        ).reshape(4, 2)

        # Convert original-image coordinates to the 224×224 training space
        corners[:, 0] *= IMAGE_SIZE / width
        corners[:, 1] *= IMAGE_SIZE / height

        image = cv2.resize(
            image,
            (IMAGE_SIZE, IMAGE_SIZE),
            interpolation=cv2.INTER_AREA
        )

        if self.augment:
            image, corners = self.apply_augmentation(
                image,
                corners
            )

        corners[:, 0] /= IMAGE_SIZE
        corners[:, 1] /= IMAGE_SIZE
        corners = np.clip(corners, 0.0, 1.0)

        image = image.astype(np.float32) / 255.0
        image = (image - MEAN) / STD
        image = torch.from_numpy(
            image.transpose(2, 0, 1)
        ).float()

        return {
            "image": image,
            "capacity": torch.tensor(
                int(row["class_index"]),
                dtype=torch.long
            ),
            "corners": torch.from_numpy(
                corners.reshape(-1)
            ).float(),
            "source_file": row["source_file"],
            "kind": row["kind"]
        }

    @staticmethod
    def apply_augmentation(image, corners):
        # Photometric augmentation
        contrast = np.random.uniform(0.85, 1.15)
        brightness = np.random.uniform(-18, 18)
        image = np.clip(
            image.astype(np.float32) * contrast + brightness,
            0,
            255
        ).astype(np.uint8)

        if np.random.rand() < 0.25:
            image = cv2.GaussianBlur(image, (3, 3), 0)

        if np.random.rand() < 0.25:
            noise = np.random.normal(
                0,
                3,
                image.shape
            ).astype(np.float32)

            image = np.clip(
                image.astype(np.float32) + noise,
                0,
                255
            ).astype(np.uint8)

        # Small geometric augmentation with matching corner transformation
        angle = np.random.uniform(-3.0, 3.0)
        scale = np.random.uniform(0.96, 1.04)
        translate_x = np.random.uniform(-6, 6)
        translate_y = np.random.uniform(-6, 6)

        matrix = cv2.getRotationMatrix2D(
            ((IMAGE_SIZE - 1) / 2, (IMAGE_SIZE - 1) / 2),
            angle,
            scale
        )

        matrix[:, 2] += [translate_x, translate_y]

        transformed_corners = cv2.transform(
            corners[None, :, :],
            matrix
        )[0]

        # Apply only geometrically valid transformations
        if (
            np.all(transformed_corners[:, 0] >= 0)
            and np.all(transformed_corners[:, 0] < IMAGE_SIZE)
            and np.all(transformed_corners[:, 1] >= 0)
            and np.all(transformed_corners[:, 1] < IMAGE_SIZE)
        ):
            image = cv2.warpAffine(
                image,
                matrix,
                (IMAGE_SIZE, IMAGE_SIZE),
                flags=cv2.INTER_LINEAR,
                borderMode=cv2.BORDER_REFLECT_101
            )

            corners = transformed_corners

        return image, corners


class BoxSizeClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = efficientnet_b0(weights=None).features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1280, 128),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 4)
        )

    def forward(self, inputs):
        return self.classifier(
            self.pool(self.features(inputs))
        )


class SpatialCornerRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = efficientnet_b0(weights=None).features
        self.regressor = nn.Sequential(
            nn.Conv2d(1280, 64, 1),
            nn.BatchNorm2d(64),
            nn.SiLU(),
            nn.AdaptiveAvgPool2d((7, 7)),
            nn.Flatten(),
            nn.Linear(3136, 256),
            nn.SiLU(),
            nn.Dropout(0.25),
            nn.Linear(256, 8),
            nn.Sigmoid()
        )

    def forward(self, inputs):
        return self.regressor(
            self.features(inputs)
        )


train_dataset = LocalizationDataset(
    dataset[dataset["split"] == "train"],
    augment=True
)

validation_dataset = LocalizationDataset(
    dataset[dataset["split"] == "validation"],
    augment=False
)

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    generator=loader_generator
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

size_model = BoxSizeClassifier()
corner_model = SpatialCornerRegressor()

size_weights = OLD_MODEL_DIR / "efficientnet_b0_box_size_classifier.pt"
corner_weights = OLD_MODEL_DIR / "efficientnet_b0_spatial_corner_regressor.pt"

size_model.load_state_dict(
    torch.load(
        size_weights,
        map_location="cpu",
        weights_only=True
    ),
    strict=True
)

corner_model.load_state_dict(
    torch.load(
        corner_weights,
        map_location="cpu",
        weights_only=True
    ),
    strict=True
)

size_model = size_model.to(DEVICE)
corner_model = corner_model.to(DEVICE)

test_batch = next(iter(train_loader))

with torch.inference_mode():
    images = test_batch["image"].to(DEVICE)
    size_output = size_model(images)
    corner_output = corner_model(images)

assert size_output.shape == (len(images), 4)
assert corner_output.shape == (len(images), 8)

print("Device:", DEVICE)
print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Size output:", tuple(size_output.shape))
print("Corner output:", tuple(corner_output.shape))
print("PASS: datasets and pretrained models are ready")

/homes/mxqasim/Chocolathon/.venv/lib64/python3.9/site-packages/networkx/utils/backends.py:135: RuntimeWarning: networkx backend defined more than once: nx-loopback
  backends.update(_get_backends("networkx.backends"))


Device: cpu
Training batches: 6
Validation batches: 2
Size output: (8, 4)
Corner output: (8, 8)
PASS: datasets and pretrained models are ready


In [3]:
import copy
import time
from torch.nn import functional as F

MAX_EPOCHS = 20
PATIENCE = 5
BACKBONE_LR = 1e-5
HEAD_LR = 2e-4
WEIGHT_DECAY = 1e-4

SIZE_CANDIDATE = RUN_DIR / "efficientnet_b0_box_size_classifier_candidate.pt"
CORNER_CANDIDATE = RUN_DIR / "efficientnet_b0_spatial_corner_regressor_candidate.pt"


def create_optimizer(model, head):
    backbone_parameters = [
        parameter
        for parameter in model.features.parameters()
        if parameter.requires_grad
    ]

    head_parameters = [
        parameter
        for parameter in head.parameters()
        if parameter.requires_grad
    ]

    return torch.optim.AdamW(
        [
            {
                "params": backbone_parameters,
                "lr": BACKBONE_LR
            },
            {
                "params": head_parameters,
                "lr": HEAD_LR
            }
        ],
        weight_decay=WEIGHT_DECAY
    )


@torch.inference_mode()
def evaluate_size(model):
    model.eval()
    loss_total = 0.0
    correct = 0
    total = 0

    for batch in validation_loader:
        images = batch["image"].to(
            DEVICE,
            non_blocking=True
        )

        targets = batch["capacity"].to(
            DEVICE,
            non_blocking=True
        )

        logits = model(images)
        loss = F.cross_entropy(
            logits,
            targets,
            label_smoothing=0.05
        )

        loss_total += loss.item() * len(images)
        correct += (
            logits.argmax(dim=1) == targets
        ).sum().item()
        total += len(images)

    return {
        "loss": loss_total / total,
        "accuracy": correct / total
    }


@torch.inference_mode()
def evaluate_corners(model):
    model.eval()
    loss_total = 0.0
    distance_total = 0.0
    corner_total = 0
    image_total = 0

    for batch in validation_loader:
        images = batch["image"].to(
            DEVICE,
            non_blocking=True
        )

        targets = batch["corners"].to(
            DEVICE,
            non_blocking=True
        )

        predictions = model(images)
        loss = F.smooth_l1_loss(
            predictions,
            targets,
            beta=0.02
        )

        prediction_points = predictions.reshape(-1, 4, 2)
        target_points = targets.reshape(-1, 4, 2)

        distances = torch.linalg.vector_norm(
            prediction_points - target_points,
            dim=2
        )

        loss_total += loss.item() * len(images)
        distance_total += distances.sum().item()
        corner_total += distances.numel()
        image_total += len(images)

    return {
        "loss": loss_total / image_total,
        "normalized_corner_error": distance_total / corner_total
    }


def train_size_model(model):
    optimizer = create_optimizer(
        model,
        model.classifier
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2
    )

    best_state = copy.deepcopy(model.state_dict())
    best_accuracy = -1.0
    best_loss = float("inf")
    patience_left = PATIENCE
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        started = time.perf_counter()
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for batch in train_loader:
            images = batch["image"].to(
                DEVICE,
                non_blocking=True
            )

            targets = batch["capacity"].to(
                DEVICE,
                non_blocking=True
            )

            optimizer.zero_grad(set_to_none=True)
            logits = model(images)

            loss = F.cross_entropy(
                logits,
                targets,
                label_smoothing=0.05
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            train_loss += loss.item() * len(images)
            train_correct += (
                logits.argmax(dim=1) == targets
            ).sum().item()
            train_total += len(images)

        validation = evaluate_size(model)
        train_accuracy = train_correct / train_total
        train_loss /= train_total
        scheduler.step(validation["accuracy"])

        improved = (
            validation["accuracy"] > best_accuracy
            or (
                validation["accuracy"] == best_accuracy
                and validation["loss"] < best_loss
            )
        )

        if improved:
            best_accuracy = validation["accuracy"]
            best_loss = validation["loss"]
            best_state = copy.deepcopy(model.state_dict())
            patience_left = PATIENCE
        else:
            patience_left -= 1

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "validation_loss": validation["loss"],
            "validation_accuracy": validation["accuracy"]
        })

        print(
            f"SIZE {epoch:02d} | "
            f"train loss={train_loss:.4f} "
            f"acc={train_accuracy:.3f} | "
            f"val loss={validation['loss']:.4f} "
            f"acc={validation['accuracy']:.3f} | "
            f"{time.perf_counter() - started:.1f}s"
        )

        if patience_left == 0:
            print("Size-model early stopping")
            break

    model.load_state_dict(best_state)
    torch.save(
        model.state_dict(),
        SIZE_CANDIDATE
    )

    return pd.DataFrame(history)


def train_corner_model(model):
    optimizer = create_optimizer(
        model,
        model.regressor
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2
    )

    best_state = copy.deepcopy(model.state_dict())
    best_error = float("inf")
    patience_left = PATIENCE
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        started = time.perf_counter()
        model.train()
        train_loss = 0.0
        train_total = 0

        for batch in train_loader:
            images = batch["image"].to(
                DEVICE,
                non_blocking=True
            )

            targets = batch["corners"].to(
                DEVICE,
                non_blocking=True
            )

            optimizer.zero_grad(set_to_none=True)
            predictions = model(images)

            loss = F.smooth_l1_loss(
                predictions,
                targets,
                beta=0.02
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            train_loss += loss.item() * len(images)
            train_total += len(images)

        train_loss /= train_total
        validation = evaluate_corners(model)
        validation_error = validation[
            "normalized_corner_error"
        ]

        scheduler.step(validation_error)

        if validation_error < best_error:
            best_error = validation_error
            best_state = copy.deepcopy(model.state_dict())
            patience_left = PATIENCE
        else:
            patience_left -= 1

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "validation_loss": validation["loss"],
            "validation_normalized_corner_error": validation_error
        })

        print(
            f"CORNER {epoch:02d} | "
            f"train loss={train_loss:.5f} | "
            f"val loss={validation['loss']:.5f} "
            f"corner={validation_error:.5f} | "
            f"{time.perf_counter() - started:.1f}s"
        )

        if patience_left == 0:
            print("Corner-model early stopping")
            break

    model.load_state_dict(best_state)
    torch.save(
        model.state_dict(),
        CORNER_CANDIDATE
    )

    return pd.DataFrame(history)


print("Training box-size classifier...")
size_history = train_size_model(size_model)

print("\nTraining spatial corner regressor...")
corner_history = train_corner_model(corner_model)

size_history.to_csv(
    RUN_DIR / "size_training_history.csv",
    index=False
)

corner_history.to_csv(
    RUN_DIR / "corner_training_history.csv",
    index=False
)

print("\nCandidate models saved:")
print(SIZE_CANDIDATE)
print(CORNER_CANDIDATE)

Training box-size classifier...
SIZE 01 | train loss=1.4215 acc=0.458 | val loss=1.8510 acc=0.625 | 13.8s
SIZE 02 | train loss=1.1911 acc=0.542 | val loss=1.3863 acc=0.625 | 8.5s
SIZE 03 | train loss=0.9374 acc=0.583 | val loss=1.0393 acc=0.812 | 8.5s
SIZE 04 | train loss=0.8193 acc=0.688 | val loss=0.7938 acc=0.812 | 8.6s
SIZE 05 | train loss=0.6444 acc=0.875 | val loss=0.6273 acc=0.875 | 8.6s
SIZE 06 | train loss=0.5919 acc=0.938 | val loss=0.5058 acc=0.938 | 8.6s
SIZE 07 | train loss=0.4474 acc=0.979 | val loss=0.4281 acc=0.938 | 8.6s
SIZE 08 | train loss=0.4314 acc=0.958 | val loss=0.3848 acc=1.000 | 8.5s
SIZE 09 | train loss=0.3670 acc=0.979 | val loss=0.3405 acc=1.000 | 8.5s
SIZE 10 | train loss=0.3113 acc=1.000 | val loss=0.3247 acc=1.000 | 8.4s
SIZE 11 | train loss=0.3112 acc=1.000 | val loss=0.3254 acc=1.000 | 8.7s
SIZE 12 | train loss=0.5250 acc=0.875 | val loss=0.3044 acc=1.000 | 8.5s
SIZE 13 | train loss=0.3052 acc=1.000 | val loss=0.2866 acc=1.000 | 8.5s
SIZE 14 | train lo

In [4]:
def load_size_model(path):
    model = BoxSizeClassifier()
    model.load_state_dict(
        torch.load(
            path,
            map_location="cpu",
            weights_only=True
        ),
        strict=True
    )
    return model.to(DEVICE).eval()


def load_corner_model(path):
    model = SpatialCornerRegressor()
    model.load_state_dict(
        torch.load(
            path,
            map_location="cpu",
            weights_only=True
        ),
        strict=True
    )
    return model.to(DEVICE).eval()


@torch.inference_mode()
def collect_size_results(model):
    rows = []

    for batch in validation_loader:
        images = batch["image"].to(DEVICE)
        targets = batch["capacity"].numpy()
        probabilities = model(
            images
        ).softmax(dim=1).cpu().numpy()

        predictions = probabilities.argmax(axis=1)

        for index in range(len(images)):
            rows.append({
                "source_file": batch["source_file"][index],
                "kind": batch["kind"][index],
                "expected_index": int(targets[index]),
                "predicted_index": int(predictions[index]),
                "expected_capacity": CAPACITIES[int(targets[index])],
                "predicted_capacity": CAPACITIES[int(predictions[index])],
                "confidence": float(probabilities[index].max()),
                "correct": int(predictions[index]) == int(targets[index])
            })

    return pd.DataFrame(rows)


@torch.inference_mode()
def collect_corner_results(model):
    rows = []

    for batch in validation_loader:
        images = batch["image"].to(DEVICE)
        targets = batch["corners"].to(DEVICE)
        predictions = model(images)

        prediction_points = predictions.reshape(-1, 4, 2)
        target_points = targets.reshape(-1, 4, 2)

        distances = torch.linalg.vector_norm(
            prediction_points - target_points,
            dim=2
        ).cpu().numpy()

        for index in range(len(images)):
            rows.append({
                "source_file": batch["source_file"][index],
                "kind": batch["kind"][index],
                "mean_normalized_error": float(
                    distances[index].mean()
                ),
                "max_normalized_error": float(
                    distances[index].max()
                )
            })

    return pd.DataFrame(rows)


old_size_model = load_size_model(size_weights)
new_size_model = load_size_model(SIZE_CANDIDATE)
old_corner_model = load_corner_model(corner_weights)
new_corner_model = load_corner_model(CORNER_CANDIDATE)

old_size_results = collect_size_results(old_size_model)
new_size_results = collect_size_results(new_size_model)
old_corner_results = collect_corner_results(old_corner_model)
new_corner_results = collect_corner_results(new_corner_model)

comparison = {
    "old_size_accuracy": float(
        old_size_results["correct"].mean()
    ),
    "new_size_accuracy": float(
        new_size_results["correct"].mean()
    ),
    "old_corner_normalized_error": float(
        old_corner_results["mean_normalized_error"].mean()
    ),
    "new_corner_normalized_error": float(
        new_corner_results["mean_normalized_error"].mean()
    )
}

comparison["size_improved"] = (
    comparison["new_size_accuracy"]
    > comparison["old_size_accuracy"]
)

comparison["corner_improved"] = (
    comparison["new_corner_normalized_error"]
    < comparison["old_corner_normalized_error"]
)

with (
    RUN_DIR / "validation_comparison.json"
).open("w") as file:
    json.dump(
        comparison,
        file,
        indent=2
    )

old_size_results.to_csv(
    RUN_DIR / "old_size_validation.csv",
    index=False
)

new_size_results.to_csv(
    RUN_DIR / "new_size_validation.csv",
    index=False
)

old_corner_results.to_csv(
    RUN_DIR / "old_corner_validation.csv",
    index=False
)

new_corner_results.to_csv(
    RUN_DIR / "new_corner_validation.csv",
    index=False
)

print("VALIDATION COMPARISON")
print(json.dumps(comparison, indent=2))

print("\nCandidate capacity predictions:")
display(
    new_size_results[
        [
            "source_file", "kind",
            "expected_capacity", "predicted_capacity",
            "confidence", "correct"
        ]
    ]
)

print("\nCorner error by held-out source:")
corner_by_source = pd.DataFrame({
    "old": old_corner_results.groupby(
        "source_file"
    )["mean_normalized_error"].mean(),
    "candidate": new_corner_results.groupby(
        "source_file"
    )["mean_normalized_error"].mean()
})

corner_by_source["improvement"] = (
    corner_by_source["old"]
    - corner_by_source["candidate"]
)

display(corner_by_source)

print("\nDo not replace deployment models yet.")
print("First review these held-out validation results.")

VALIDATION COMPARISON
{
  "old_size_accuracy": 0.625,
  "new_size_accuracy": 1.0,
  "old_corner_normalized_error": 0.04928206722252071,
  "new_corner_normalized_error": 0.061967486748471856,
  "size_improved": true,
  "corner_improved": false
}

Candidate capacity predictions:


,source_file,kind,expected_capacity,predicted_capacity,confidence,correct
0,IMG_2021.jpg,original_empty,6,6,0.994599,True
1,IMG_2026.jpg,original_empty,16,16,0.726326,True
2,IMG_2030.jpg,original_empty,30,30,0.838910,True
3,IMG_2034.jpg,original_empty,50,50,0.696911,True
4,IMG_2021.jpg,synthetic_filled,6,6,0.988165,True
5,IMG_2021.jpg,synthetic_filled,6,6,0.991733,True
6,IMG_2021.jpg,synthetic_filled,6,6,0.985916,True
7,IMG_2026.jpg,synthetic_filled,16,16,0.955839,True
8,IMG_2026.jpg,synthetic_filled,16,16,0.974201,True
9,IMG_2026.jpg,synthetic_filled,16,16,0.944471,True



Corner error by held-out source:


,old,candidate,improvement
source_file,,,
IMG_2021.jpg,0.011482,0.055701,-0.044219
IMG_2026.jpg,0.019372,0.040956,-0.021584
IMG_2030.jpg,0.099618,0.093715,0.005903
IMG_2034.jpg,0.066656,0.057498,0.009158



Do not replace deployment models yet.
First review these held-out validation results.


In [5]:
# Cell 5 — Create the localization-v2 deployment bundle

from pathlib import Path
import json
import shutil

PROJECT_ROOT = Path("/homes/mxqasim/Chocolathon")

OLD_DIR = (
    PROJECT_ROOT
    / "data/experiments/box_localization_v1/outputs"
)

V2_DIR = (
    PROJECT_ROOT
    / "data/experiments/box_localization_v2"
)

DEPLOY_DIR = V2_DIR / "deployment"
DEPLOY_DIR.mkdir(parents=True, exist_ok=True)

source_files = {
    # New classifier: validated at 100%
    "size_classifier": (
        V2_DIR
        / "efficientnet_b0_box_size_classifier_candidate.pt"
    ),

    # Old regressor remains best for small boxes
    "corner_small": (
        OLD_DIR
        / "efficientnet_b0_spatial_corner_regressor.pt"
    ),

    # New regressor improves difficult large boxes
    "corner_large": (
        V2_DIR
        / "efficientnet_b0_spatial_corner_regressor_candidate.pt"
    ),
}

for name, path in source_files.items():
    assert path.exists(), f"Missing {name}: {path}"

deployment_files = {
    "size_classifier":
        DEPLOY_DIR / "efficientnet_b0_box_size_classifier.pt",

    "corner_small":
        DEPLOY_DIR / "efficientnet_b0_corner_regressor_small.pt",

    "corner_large":
        DEPLOY_DIR / "efficientnet_b0_corner_regressor_large.pt",
}

for name, destination in deployment_files.items():
    shutil.copy2(source_files[name], destination)

config = {
    "version": "box_localization_v2_routed",
    "capacities": [6, 16, 30, 50],
    "capacity_class_order": [6, 16, 30, 50],

    "size_classifier": deployment_files[
        "size_classifier"
    ].name,

    "corner_routing": {
        "6": deployment_files["corner_small"].name,
        "16": deployment_files["corner_small"].name,
        "30": deployment_files["corner_large"].name,
        "50": deployment_files["corner_large"].name,
    },

    "validation": {
        "old_size_accuracy": 0.625,
        "new_size_accuracy": 1.0,
        "old_corner_normalized_error": 0.04928206722252071,
        "new_corner_normalized_error": 0.061967486748471856,
        "hybrid_corner_normalized_error": 0.04551775,
    },

    "notes": [
        "The new size classifier is used for all capacities.",
        "The old corner regressor is used for 6-slot and 16-slot boxes.",
        "The new corner regressor is used for 30-slot and 50-slot boxes.",
        "The original deployment artifacts were not overwritten.",
    ],
}

config_path = DEPLOY_DIR / "localization_config.json"

with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print("PASS: routed localization bundle created")
print("Deployment directory:", DEPLOY_DIR)

for name, path in deployment_files.items():
    print(f"{name:20s} {path.name:52s} {path.stat().st_size / 1024**2:.2f} MB")

print("config:", config_path.name)

PASS: routed localization bundle created
Deployment directory: /homes/mxqasim/Chocolathon/data/experiments/box_localization_v2/deployment
size_classifier      efficientnet_b0_box_size_classifier.pt               16.21 MB
corner_small         efficientnet_b0_corner_regressor_small.pt            18.97 MB
corner_large         efficientnet_b0_corner_regressor_large.pt            18.97 MB
config: localization_config.json


In [6]:
# Cell 6 — Validate deployment artifacts and model routing

from pathlib import Path
import json
import torch

PROJECT_ROOT = Path("/homes/mxqasim/Chocolathon")

DEPLOY_DIR = (
    PROJECT_ROOT
    / "data/experiments/box_localization_v2/deployment"
)

CONFIG_PATH = DEPLOY_DIR / "localization_config.json"

assert CONFIG_PATH.exists(), f"Missing config: {CONFIG_PATH}"

with open(CONFIG_PATH) as f:
    deployment_config = json.load(f)

required_files = {
    "size_classifier":
        DEPLOY_DIR / deployment_config["size_classifier"],

    "corner_small":
        DEPLOY_DIR
        / deployment_config["corner_routing"]["6"],

    "corner_large":
        DEPLOY_DIR
        / deployment_config["corner_routing"]["30"],
}

loaded_states = {}

for name, path in required_files.items():
    assert path.exists(), f"Missing deployment file: {path}"

    state = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    assert isinstance(state, dict), (
        f"{path.name} does not contain a state dictionary"
    )

    loaded_states[name] = state

    print(
        f"✓ {name:18s}"
        f"{path.name:52s}"
        f"{len(state):4d} tensors"
    )

# Both regressors must have exactly compatible architectures.
small_keys = set(loaded_states["corner_small"].keys())
large_keys = set(loaded_states["corner_large"].keys())

assert small_keys == large_keys, (
    "Small and large corner regressors have incompatible keys"
)

shape_errors = []

for key in sorted(small_keys):
    small_shape = tuple(loaded_states["corner_small"][key].shape)
    large_shape = tuple(loaded_states["corner_large"][key].shape)

    if small_shape != large_shape:
        shape_errors.append(
            f"{key}: small={small_shape}, large={large_shape}"
        )

assert not shape_errors, "\n".join(shape_errors)

print("\nCapacity routing:")

for capacity in deployment_config["capacities"]:
    model_name = deployment_config["corner_routing"][str(capacity)]
    print(f"  {capacity:2d} slots -> {model_name}")

assert (
    deployment_config["corner_routing"]["6"]
    == deployment_config["corner_routing"]["16"]
)

assert (
    deployment_config["corner_routing"]["30"]
    == deployment_config["corner_routing"]["50"]
)

assert (
    deployment_config["corner_routing"]["6"]
    != deployment_config["corner_routing"]["30"]
)

print("\nPASS: localization-v2 deployment bundle is valid")
print("No original model was overwritten.")

✓ size_classifier   efficientnet_b0_box_size_classifier.pt               362 tensors
✓ corner_small      efficientnet_b0_corner_regressor_small.pt            369 tensors
✓ corner_large      efficientnet_b0_corner_regressor_large.pt            369 tensors

Capacity routing:
   6 slots -> efficientnet_b0_corner_regressor_small.pt
  16 slots -> efficientnet_b0_corner_regressor_small.pt
  30 slots -> efficientnet_b0_corner_regressor_large.pt
  50 slots -> efficientnet_b0_corner_regressor_large.pt

PASS: localization-v2 deployment bundle is valid
No original model was overwritten.


In [1]:
# Cell 7 — Memory-safe corner refinement
import cv2, json, math, csv
import numpy as np
from pathlib import Path

def line_intersection(a, b):
    x1, y1, x2, y2 = map(float, a)
    x3, y3, x4, y4 = map(float, b)
    denominator = (x1-x2)*(y3-y4)-(y1-y2)*(x3-x4)
    if abs(denominator) < 1e-6: return None
    px = ((x1*y2-y1*x2)*(x3-x4)-(x1-x2)*(x3*y4-y3*x4))/denominator
    py = ((x1*y2-y1*x2)*(y3-y4)-(y1-y2)*(x3*y4-y3*x4))/denominator
    return np.array([px, py], np.float32)

def extract_edge_segments(rgb):
    h0, w0 = rgb.shape[:2]
    scale = min(1.0, 1400/max(h0, w0))
    image = cv2.resize(rgb, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA) if scale < 1 else rgb
    h, w = image.shape[:2]
    diagonal = np.hypot(w, h)
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    gray = cv2.createCLAHE(2.0, (8, 8)).apply(gray)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    median = np.median(gray)
    edges = cv2.Canny(gray, int(max(10, .55*median)), int(min(255, 1.45*median)))
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=40, minLineLength=max(35, int(.07*diagonal)), maxLineGap=max(12, int(.025*diagonal)))
    segments = np.empty((0, 4), np.float32) if lines is None else np.asarray(lines, np.float32).reshape(-1, 4)
    return {"segments": segments, "scale": scale, "diagonal": diagonal, "height": h0, "width": w0}

def refine_from_segments(predicted, edge_data, band_fraction=.08, angle_tolerance=15):
    original = np.asarray(predicted, np.float32).reshape(4, 2)
    segments = edge_data["segments"]
    if not len(segments): return original, {"accepted": False, "reason": "no_lines"}

    scale, diagonal = edge_data["scale"], edge_data["diagonal"]
    points = original*scale
    cosine_limit = math.cos(math.radians(angle_tolerance))
    selected, band = [], band_fraction*diagonal

    for index in range(4):
        start, end = points[index], points[(index+1) % 4]
        vector = end-start
        length = np.linalg.norm(vector)
        if length < 2: return original, {"accepted": False, "reason": "short_edge"}

        direction = vector/length
        normal = np.array([-direction[1], direction[0]], np.float32)
        best_line, best_score = None, -np.inf

        for segment_values in segments:
            x1, y1, x2, y2 = segment_values
            p1, p2 = np.array([x1, y1], np.float32), np.array([x2, y2], np.float32)
            segment = p2-p1
            segment_length = np.linalg.norm(segment)
            if segment_length < 20 or abs(np.dot(segment/segment_length, direction)) < cosine_limit: continue

            distance = abs(np.dot((p1+p2)/2-start, normal))
            if distance > band: continue

            projections = sorted([np.dot(p1-start, direction), np.dot(p2-start, direction)])
            overlap = max(0, min(length, projections[1])-max(0, projections[0]))/length
            if overlap < .20: continue

            score = segment_length*(.5+overlap)-1.5*distance
            if score > best_score: best_line, best_score = segment_values.copy(), score

        if best_line is None: return original, {"accepted": False, "reason": f"edge_{index}_missing"}
        selected.append(best_line)

    refined = [
        line_intersection(selected[3], selected[0]),
        line_intersection(selected[0], selected[1]),
        line_intersection(selected[1], selected[2]),
        line_intersection(selected[2], selected[3]),
    ]
    if any(point is None for point in refined): return original, {"accepted": False, "reason": "parallel_lines"}

    refined = np.asarray(refined, np.float32)/scale
    original_area, refined_area = abs(cv2.contourArea(original)), abs(cv2.contourArea(refined))
    valid = np.isfinite(refined).all() and cv2.isContourConvex(refined)
    valid &= .55*original_area <= refined_area <= 1.55*original_area
    valid &= np.all(refined[:, 0] >= 0) and np.all(refined[:, 0] < edge_data["width"])
    valid &= np.all(refined[:, 1] >= 0) and np.all(refined[:, 1] < edge_data["height"])

    if not valid: return original, {"accepted": False, "reason": "invalid_geometry"}
    return refined, {"accepted": True, "reason": "refined"}

print("PASS: memory-safe corner refinement ready")

PASS: memory-safe corner refinement ready


In [3]:
# Cell 8 — Select parameters on training sources and evaluate v4
import pandas as pd
from chocolathon_inference import Chocolathon, CORNER_NAMES
from diagnose_chocolathon_pipeline import resolve_path, parse_corners, load_truth, score

ROOT = Path("/homes/mxqasim/Chocolathon")
SYNTH_ROOT = ROOT/"data/synthetic_filled_boxes/v4"
OUTPUT = ROOT/"outputs/corner_refinement_v2"
HELD_OUT = {"IMG_2021.jpg", "IMG_2026.jpg", "IMG_2030.jpg", "IMG_2034.jpg"}
CONFIGS = [(band, angle) for band in (.04, .06, .08, .10) for angle in (10, 15, 20)]

with (SYNTH_ROOT/"manifest.csv").open(newline="") as f:
    manifest = list(csv.DictReader(f))

engine = Chocolathon(ROOT, "cpu", 1, 8, "original", "uniform")
samples = []

for index, item in enumerate(manifest, 1):
    image = resolve_path(item.get("image") or item.get("image_path") or item.get("file"), SYNTH_ROOT)
    gt_path = resolve_path(item.get("gt_path") or item.get("slots_csv") or item.get("ground_truth"), SYNTH_ROOT)
    gt_rows, truth = load_truth(gt_path)
    expected_capacity = int(item.get("box_size") or gt_rows[0].get("box_size"))
    expected_corners = np.asarray([parse_corners(item)[name] for name in CORNER_NAMES], np.float32)

    prediction = engine.predict(image)
    automatic, rgb = prediction[0], prediction[1]
    predicted_corners = np.asarray(automatic["corners"], np.float32)
    edge_data = extract_edge_segments(rgb)
    source = item.get("source_file") or image.name.split("_size")[0]+".jpg"

    samples.append({
        "image": image, "source": source, "truth": truth, "expected_capacity": expected_capacity,
        "expected_corners": expected_corners, "predicted_corners": predicted_corners,
        "automatic": automatic, "edge_data": edge_data,
    })
    del prediction, rgb
    print(f"[{index:02d}/{len(manifest)}] prepared {image.name}")

config_rows = []
for band, angle in CONFIGS:
    errors, accepted = [], 0

    for sample in samples:
        if sample["source"] in HELD_OUT: continue
        refined, info = refine_from_segments(sample["predicted_corners"], sample["edge_data"], band, angle)
        diagonal = np.hypot(sample["edge_data"]["width"], sample["edge_data"]["height"])
        errors.append(np.linalg.norm(refined-sample["expected_corners"], axis=1).mean()/diagonal)
        accepted += int(info["accepted"])

    config_rows.append({"band": band, "angle": angle, "train_corner_error": np.mean(errors), "accepted": accepted})

config_table = pd.DataFrame(config_rows).sort_values(["train_corner_error", "accepted"], ascending=[True, False]).reset_index(drop=True)
best_band, best_angle = float(config_table.iloc[0]["band"]), int(config_table.iloc[0]["angle"])

print("\nParameter comparison:")
display(config_table)
print(f"Selected from training sources: band={best_band}, angle={best_angle}")

rows = []
for index, sample in enumerate(samples, 1):
    refined, info = refine_from_segments(sample["predicted_corners"], sample["edge_data"], best_band, best_angle)
    h, w = sample["edge_data"]["height"], sample["edge_data"]["width"]
    annotation = {
        "image_width": w, "image_height": h, "box_size": sample["automatic"]["predicted_capacity"],
        "corners": {name: refined[i].tolist() for i, name in enumerate(CORNER_NAMES)},
    }

    refined_prediction = engine.predict(sample["image"], annotation=annotation)
    refined_result = refined_prediction[0]
    raw_score, refined_score = score(sample["automatic"], sample["truth"]), score(refined_result, sample["truth"])
    diagonal = np.hypot(w, h)
    raw_error = np.linalg.norm(sample["predicted_corners"]-sample["expected_corners"], axis=1).mean()/diagonal
    refined_error = np.linalg.norm(refined-sample["expected_corners"], axis=1).mean()/diagonal

    rows.append({
        "image": sample["image"].name, "source": sample["source"],
        "split": "validation" if sample["source"] in HELD_OUT else "train",
        "capacity": sample["expected_capacity"], "accepted": info["accepted"],
        "raw_corner_error": raw_error, "refined_corner_error": refined_error,
        "raw_flavor_correct": raw_score["flavor_correct"], "refined_flavor_correct": refined_score["flavor_correct"],
        "expected_occupied": raw_score["expected_occupied"],
        "raw_false_empty": raw_score["false_empty"], "refined_false_empty": refined_score["false_empty"],
        "raw_count_error": raw_score["count_abs_error"], "refined_count_error": refined_score["count_abs_error"],
    })
    del refined_prediction
    print(f'[{index:02d}/{len(samples)}] {sample["image"].name}: accepted={info["accepted"]} '
          f'corner={raw_error:.4f}->{refined_error:.4f} flavor={raw_score["flavor_correct"]}->{refined_score["flavor_correct"]}')

results = pd.DataFrame(rows)
summary = {}

for split in ("all", "train", "validation"):
    part = results if split == "all" else results[results["split"] == split]
    summary[split] = {
        "images": len(part), "accepted": int(part["accepted"].sum()),
        "raw_corner_error": float(part["raw_corner_error"].mean()),
        "refined_corner_error": float(part["refined_corner_error"].mean()),
        "raw_flavor_accuracy": float(part["raw_flavor_correct"].sum()/part["expected_occupied"].sum()),
        "refined_flavor_accuracy": float(part["refined_flavor_correct"].sum()/part["expected_occupied"].sum()),
        "raw_false_empty": int(part["raw_false_empty"].sum()),
        "refined_false_empty": int(part["refined_false_empty"].sum()),
        "raw_count_error": int(part["raw_count_error"].sum()),
        "refined_count_error": int(part["refined_count_error"].sum()),
    }

OUTPUT.mkdir(parents=True, exist_ok=True)
results.to_csv(OUTPUT/"results.csv", index=False)
(OUTPUT/"summary.json").write_text(json.dumps(summary, indent=2))

print("\nCORNER REFINEMENT SUMMARY")
print(json.dumps(summary, indent=2))

/homes/mxqasim/Chocolathon/.venv/lib64/python3.9/site-packages/networkx/utils/backends.py:135: RuntimeWarning: networkx backend defined more than once: nx-loopback
  backends.update(_get_backends("networkx.backends"))


[01/48] prepared IMG_2019_size6_render00.jpg
[02/48] prepared IMG_2019_size6_render01.jpg
[03/48] prepared IMG_2019_size6_render02.jpg
[04/48] prepared IMG_2020_size6_render00.jpg
[05/48] prepared IMG_2020_size6_render01.jpg
[06/48] prepared IMG_2020_size6_render02.jpg
[07/48] prepared IMG_2021_size6_render00.jpg
[08/48] prepared IMG_2021_size6_render01.jpg
[09/48] prepared IMG_2021_size6_render02.jpg
[10/48] prepared IMG_2022_size16_render00.jpg
[11/48] prepared IMG_2022_size16_render01.jpg
[12/48] prepared IMG_2022_size16_render02.jpg
[13/48] prepared IMG_2023_size16_render00.jpg
[14/48] prepared IMG_2023_size16_render01.jpg
[15/48] prepared IMG_2023_size16_render02.jpg
[16/48] prepared IMG_2024_size16_render00.jpg
[17/48] prepared IMG_2024_size16_render01.jpg
[18/48] prepared IMG_2024_size16_render02.jpg
[19/48] prepared IMG_2025_size16_render00.jpg
[20/48] prepared IMG_2025_size16_render01.jpg
[21/48] prepared IMG_2025_size16_render02.jpg
[22/48] prepared IMG_2026_size16_render00.j

,band,angle,train_corner_error,accepted
0,0.04,10,0.041029,32
1,0.04,15,0.042060,33
2,0.06,15,0.042863,35
3,0.06,10,0.042915,35
4,0.08,15,0.045685,33
5,0.08,10,0.046201,34
6,0.06,20,0.047408,32
7,0.04,20,0.049226,34
8,0.08,20,0.052074,32
9,0.10,15,0.053338,31


Selected from training sources: band=0.04, angle=10
[01/48] IMG_2019_size6_render00.jpg: accepted=True corner=0.0365->0.0318 flavor=6->5
[02/48] IMG_2019_size6_render01.jpg: accepted=True corner=0.0307->0.0251 flavor=6->6
[03/48] IMG_2019_size6_render02.jpg: accepted=True corner=0.0309->0.0439 flavor=6->6
[04/48] IMG_2020_size6_render00.jpg: accepted=True corner=0.0406->0.0287 flavor=5->6
[05/48] IMG_2020_size6_render01.jpg: accepted=True corner=0.0453->0.0287 flavor=5->6
[06/48] IMG_2020_size6_render02.jpg: accepted=True corner=0.0441->0.0273 flavor=4->6
[07/48] IMG_2021_size6_render00.jpg: accepted=True corner=0.0075->0.0192 flavor=6->6
[08/48] IMG_2021_size6_render01.jpg: accepted=True corner=0.0069->0.0303 flavor=6->6
[09/48] IMG_2021_size6_render02.jpg: accepted=True corner=0.0085->0.0214 flavor=6->6
[10/48] IMG_2022_size16_render00.jpg: accepted=True corner=0.0211->0.0438 flavor=16->10
[11/48] IMG_2022_size16_render01.jpg: accepted=True corner=0.0237->0.0378 flavor=16->5
[12/48] 

In [4]:
# Cell 9 — Build source-separated 30/50 corner dataset with geometric augmentation
import csv, json, random, time
from copy import deepcopy
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn import functional as F

from chocolathon_inference import SpatialCornerRegressor, CORNER_NAMES, MEAN, STD, read_rgb
from diagnose_chocolathon_pipeline import resolve_path, parse_corners

SEED = 418
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
ROOT = Path("/homes/mxqasim/Chocolathon")
BOX_ROOT = ROOT/"data/boxes"
SYNTH_ROOT = ROOT/"data/synthetic_filled_boxes/v4"
OUTPUT = ROOT/"data/experiments/box_localization_v3"
CURRENT_LARGE = ROOT/"data/experiments/box_localization_v2/deployment/efficientnet_b0_corner_regressor_large.pt"
HELD_OUT = {"IMG_2030.jpg", "IMG_2034.jpg"}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT.mkdir(parents=True, exist_ok=True)

def resolve_local(value, *bases):
    path = Path(str(value))
    candidates = [path] if path.is_absolute() else [base/path for base in bases]
    for candidate in candidates:
        if candidate.exists(): return candidate.resolve()
    raise FileNotFoundError(value)

def annotation_corners(path):
    data = json.loads(path.read_text())
    values = data.get("corners", data)
    if not all(name in values for name in CORNER_NAMES): raise ValueError(f"No corners in {path}")
    return np.asarray([values[name] for name in CORNER_NAMES], np.float32)

records = []
with (BOX_ROOT/"manifest.csv").open(newline="") as f:
    boxes = list(csv.DictReader(f))

for row in boxes:
    capacity = int(row["box_size"])
    if capacity not in (30, 50): continue
    source = row["file"]
    image = BOX_ROOT/"raw"/source
    annotation_ref = row.get("annotation_file") or row.get("annotation_path") or row.get("corners_json")
    if not annotation_ref: raise ValueError(f"Missing annotation path for {source}")
    annotation = resolve_local(annotation_ref, BOX_ROOT, ROOT)
    records.append({"image": image.resolve(), "source": source, "capacity": capacity, "kind": "original_empty",
                    "corners": annotation_corners(annotation), "split": "validation" if source in HELD_OUT else "train"})

with (SYNTH_ROOT/"manifest.csv").open(newline="") as f:
    synthetic = list(csv.DictReader(f))

for row in synthetic:
    capacity = int(row["box_size"])
    if capacity not in (30, 50): continue
    image = resolve_path(row.get("image") or row.get("image_path") or row.get("file"), SYNTH_ROOT)
    source = row.get("source_file") or image.name.split("_size")[0]+".jpg"
    corners = np.asarray([parse_corners(row)[name] for name in CORNER_NAMES], np.float32)
    records.append({"image": image, "source": source, "capacity": capacity, "kind": "synthetic_filled",
                    "corners": corners, "split": "validation" if source in HELD_OUT else "train"})

records = pd.DataFrame(records)
assert len(records) == 32
assert set(records[records.split == "train"].source).isdisjoint(set(records[records.split == "validation"].source))
assert set(records.capacity) == {30, 50}
assert CURRENT_LARGE.exists()

def geometric_augment(image, corners):
    frame = np.float32([[0, 0], [223, 0], [223, 223], [0, 223]])
    for _ in range(12):
        angle, scale = random.uniform(-12, 12), random.uniform(.88, 1.12)
        affine = np.vstack([cv2.getRotationMatrix2D((111.5, 111.5), angle, scale), [0, 0, 1]]).astype(np.float32)
        affine[0, 2] += random.uniform(-13, 13)
        affine[1, 2] += random.uniform(-13, 13)
        jitter = np.random.uniform(-9, 9, (4, 2)).astype(np.float32)
        perspective = cv2.getPerspectiveTransform(frame, frame+jitter)
        matrix = perspective@affine
        transformed = cv2.perspectiveTransform(corners[None], matrix)[0]

        if np.isfinite(transformed).all() and cv2.isContourConvex(transformed):
            if np.all((transformed >= 2) & (transformed <= 221)) and abs(cv2.contourArea(transformed)) > 500:
                image = cv2.warpPerspective(image, matrix, (224, 224), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)
                corners = transformed
                break

    contrast, brightness = random.uniform(.82, 1.18), random.uniform(-18, 18)
    image = np.clip(image.astype(np.float32)*contrast+brightness, 0, 255)
    if random.random() < .35: image = cv2.GaussianBlur(image, random.choice([(3, 3), (5, 5)]), 0)
    if random.random() < .35: image += np.random.normal(0, random.uniform(1, 5), image.shape)
    return np.clip(image, 0, 255).astype(np.uint8), corners

class LargeCornerDataset(Dataset):
    def __init__(self, table, training=False, repeats=1):
        self.rows = table.to_dict("records")
        self.training, self.repeats = training, repeats

    def __len__(self): return len(self.rows)*self.repeats

    def __getitem__(self, index):
        row = self.rows[index % len(self.rows)]
        image = read_rgb(row["image"])
        h, w = image.shape[:2]
        image = cv2.resize(image, (224, 224), interpolation=cv2.INTER_AREA)
        corners = row["corners"]*np.array([223/max(w-1, 1), 223/max(h-1, 1)], np.float32)
        if self.training: image, corners = geometric_augment(image, corners)
        tensor = torch.from_numpy(((image.astype(np.float32)/255-MEAN)/STD).transpose(2, 0, 1)).float()
        target = torch.from_numpy((corners/223).reshape(-1)).float()
        return tensor, target, int(row["capacity"])

train_data = LargeCornerDataset(records[records.split == "train"], training=True, repeats=8)
validation_data = LargeCornerDataset(records[records.split == "validation"])
train_loader = DataLoader(train_data, batch_size=8, shuffle=True, num_workers=0)
validation_loader = DataLoader(validation_data, batch_size=8, shuffle=False, num_workers=0)

model = SpatialCornerRegressor()
model.load_state_dict(torch.load(CURRENT_LARGE, map_location="cpu", weights_only=True), strict=True)
model = model.to(DEVICE)

for parameter in model.features.parameters(): parameter.requires_grad = False
for block in list(model.features.children())[-2:]:
    for parameter in block.parameters(): parameter.requires_grad = True

images, targets, capacities = next(iter(train_loader))
with torch.inference_mode(): outputs = model(images.to(DEVICE)).cpu()

print("Device:", DEVICE)
print("Records:", len(records), "| Training:", sum(records.split == "train"), "| Validation:", sum(records.split == "validation"))
print(records.groupby(["split", "capacity", "kind"]).size())
print("Training samples per epoch:", len(train_data))
print("Input:", tuple(images.shape), "| Target:", tuple(targets.shape), "| Output:", tuple(outputs.shape))
print("PASS: augmented large-box dataset and model are ready")

Device: cpu
Records: 32 | Training: 24 | Validation: 8
split       capacity  kind            
train       30        original_empty      3
                      synthetic_filled    9
            50        original_empty      3
                      synthetic_filled    9
validation  30        original_empty      1
                      synthetic_filled    3
            50        original_empty      1
                      synthetic_filled    3
dtype: int64
Training samples per epoch: 192
Input: (8, 3, 224, 224) | Target: (8, 8) | Output: (8, 8)
PASS: augmented large-box dataset and model are ready


In [5]:
# Cell 10 — Fine-tune large-box corner model and retain it only if validation improves
MAX_EPOCHS, PATIENCE = 25, 7
CANDIDATE = OUTPUT/"efficientnet_b0_large_corner_augmented_candidate.pt"
HISTORY = OUTPUT/"training_history.csv"

def evaluate_corner(model, loader):
    model.eval()
    errors, losses, by_capacity = [], [], {30: [], 50: []}
    with torch.inference_mode():
        for images, targets, capacities in loader:
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            predictions = model(images)
            losses.append(F.smooth_l1_loss(predictions, targets, beta=.02).item())
            distances = torch.linalg.vector_norm(predictions.reshape(-1, 4, 2)-targets.reshape(-1, 4, 2), dim=2).mean(1).cpu().numpy()
            errors.extend(distances.tolist())
            for capacity, distance in zip(capacities.tolist(), distances.tolist()): by_capacity[capacity].append(distance)
    return {"loss": float(np.mean(losses)), "error": float(np.mean(errors)),
            "error_30": float(np.mean(by_capacity[30])), "error_50": float(np.mean(by_capacity[50]))}

baseline = evaluate_corner(model, validation_loader)
best_error, best_state = baseline["error"], deepcopy(model.state_dict())
history, patience_left = [], PATIENCE

feature_parameters = [p for p in model.features.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW([
    {"params": feature_parameters, "lr": 1e-5},
    {"params": model.regressor.parameters(), "lr": 1.5e-4},
], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=.4, patience=2, min_lr=1e-7)

print("Current large-model validation:", baseline)
print("\nTraining augmented 30/50 corner model...")

for epoch in range(1, MAX_EPOCHS+1):
    started, running_loss, batches = time.perf_counter(), 0.0, 0
    model.train()

    for images, targets, _ in train_loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        predictions = model(images)
        coordinate_loss = F.smooth_l1_loss(predictions, targets, beta=.02)
        predicted_points, target_points = predictions.reshape(-1, 4, 2), targets.reshape(-1, 4, 2)
        edge_loss = F.smooth_l1_loss(torch.roll(predicted_points, -1, 1)-predicted_points,
                                     torch.roll(target_points, -1, 1)-target_points, beta=.02)
        loss = coordinate_loss+.20*edge_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        optimizer.step()
        running_loss += loss.item()
        batches += 1

    validation = evaluate_corner(model, validation_loader)
    scheduler.step(validation["error"])
    row = {"epoch": epoch, "train_loss": running_loss/batches, **validation,
           "lr_head": optimizer.param_groups[1]["lr"], "seconds": time.perf_counter()-started}
    history.append(row)

    print(f'EPOCH {epoch:02d} | train={row["train_loss"]:.5f} | val={validation["loss"]:.5f} '
          f'corner={validation["error"]:.5f} | 30={validation["error_30"]:.5f} '
          f'50={validation["error_50"]:.5f} | {row["seconds"]:.1f}s')

    if validation["error"] < best_error-1e-5:
        best_error, best_state, patience_left = validation["error"], deepcopy(model.state_dict()), PATIENCE
    else:
        patience_left -= 1
        if patience_left == 0:
            print("Early stopping")
            break

model.load_state_dict(best_state)
final_metrics = evaluate_corner(model, validation_loader)
improved = final_metrics["error"] < baseline["error"]

pd.DataFrame(history).to_csv(HISTORY, index=False)
comparison = {
    "baseline": baseline, "candidate": final_metrics, "improved": improved,
    "relative_improvement": (baseline["error"]-final_metrics["error"])/baseline["error"],
}
(OUTPUT/"validation_comparison.json").write_text(json.dumps(comparison, indent=2))

if improved:
    torch.save(model.state_dict(), CANDIDATE)
    print("\nPASS: improved candidate saved:", CANDIDATE)
else:
    if CANDIDATE.exists(): CANDIDATE.unlink()
    print("\nREJECT: candidate did not beat the current large model; no candidate was saved.")

print(json.dumps(comparison, indent=2))
print("Deployment model was not modified.")

Current large-model validation: {'loss': 0.0405786857008934, 'error': 0.07846654206514359, 'error_30': 0.09803478047251701, 'error_50': 0.05889830365777016}

Training augmented 30/50 corner model...
EPOCH 01 | train=0.05241 | val=0.04570 corner=0.08646 | 30=0.11168 50=0.06125 | 19.2s
EPOCH 02 | train=0.04113 | val=0.03906 corner=0.07465 | 30=0.08838 50=0.06092 | 18.5s
EPOCH 03 | train=0.03643 | val=0.03456 corner=0.06816 | 30=0.07903 50=0.05728 | 18.5s
EPOCH 04 | train=0.03355 | val=0.03632 corner=0.07156 | 30=0.08575 50=0.05738 | 18.6s
EPOCH 05 | train=0.03233 | val=0.03123 corner=0.06329 | 30=0.07970 50=0.04688 | 18.5s
EPOCH 06 | train=0.03064 | val=0.03162 corner=0.06374 | 30=0.07960 50=0.04788 | 18.4s
EPOCH 07 | train=0.03014 | val=0.03462 corner=0.06784 | 30=0.08800 50=0.04768 | 18.4s
EPOCH 08 | train=0.02692 | val=0.03078 corner=0.06307 | 30=0.08260 50=0.04353 | 18.5s
EPOCH 09 | train=0.02607 | val=0.03189 corner=0.06483 | 30=0.07863 50=0.05102 | 18.6s
EPOCH 10 | train=0.02671 | 

In [6]:
# Cell 11 — Compare current and augmented large models end-to-end on v4
import csv, json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch

from chocolathon_inference import Chocolathon, SpatialCornerRegressor, CORNER_NAMES
from diagnose_chocolathon_pipeline import resolve_path, parse_corners, load_truth, score

ROOT = Path("/homes/mxqasim/Chocolathon")
SYNTH_ROOT = ROOT/"data/synthetic_filled_boxes/v4"
OUTPUT = ROOT/"outputs/large_corner_v3_evaluation"
CANDIDATE = ROOT/"data/experiments/box_localization_v3/efficientnet_b0_large_corner_augmented_candidate.pt"
HELD_OUT = {"IMG_2030.jpg", "IMG_2034.jpg"}
OUTPUT.mkdir(parents=True, exist_ok=True)
assert CANDIDATE.exists()

with (SYNTH_ROOT/"manifest.csv").open(newline="") as f:
    manifest = list(csv.DictReader(f))

engine = Chocolathon(ROOT, "cpu", 1, 8, "original", "uniform")
large_filename = engine.corner_routing[30]
assert large_filename == engine.corner_routing[50]

def evaluate_engine(engine, label):
    rows = []
    for index, item in enumerate(manifest, 1):
        image = resolve_path(item.get("image") or item.get("image_path") or item.get("file"), SYNTH_ROOT)
        gt_path = resolve_path(item.get("gt_path") or item.get("slots_csv") or item.get("ground_truth"), SYNTH_ROOT)
        gt_rows, truth = load_truth(gt_path)
        expected_capacity = int(item.get("box_size") or gt_rows[0].get("box_size"))
        expected_corners = np.asarray([parse_corners(item)[name] for name in CORNER_NAMES], np.float32)
        source = item.get("source_file") or image.name.split("_size")[0]+".jpg"

        prediction = engine.predict(image)
        result, rgb = prediction[0], prediction[1]
        predicted_corners = np.asarray(result["corners"], np.float32)
        corner_error = np.linalg.norm(predicted_corners-expected_corners, axis=1).mean()/np.hypot(*rgb.shape[:2])
        metrics = score(result, truth)

        rows.append({
            "system": label, "image": image.name, "source": source,
            "split": "validation" if source in HELD_OUT else "train",
            "capacity": expected_capacity, "capacity_correct": result["predicted_capacity"] == expected_capacity,
            "corner_error": float(corner_error), "expected_occupied": metrics["expected_occupied"],
            "flavor_correct": metrics["flavor_correct"], "false_empty": metrics["false_empty"],
            "count_error": metrics["count_abs_error"], "exact_counts": metrics["exact_flavor_counts"],
        })
        del prediction, rgb
        print(f'[{label} {index:02d}/{len(manifest)}] {image.name}')

    return pd.DataFrame(rows)

current_results = evaluate_engine(engine, "current_v2")

candidate_model = SpatialCornerRegressor()
candidate_model.load_state_dict(torch.load(CANDIDATE, map_location="cpu", weights_only=True), strict=True)
engine.corner_models[large_filename] = candidate_model.to(engine.device).eval()
candidate_results = evaluate_engine(engine, "candidate_v3")

results = pd.concat([current_results, candidate_results], ignore_index=True)
results.to_csv(OUTPUT/"per_image_results.csv", index=False)
print("PASS: both pipelines evaluated")

[current_v2 01/48] IMG_2019_size6_render00.jpg
[current_v2 02/48] IMG_2019_size6_render01.jpg
[current_v2 03/48] IMG_2019_size6_render02.jpg
[current_v2 04/48] IMG_2020_size6_render00.jpg
[current_v2 05/48] IMG_2020_size6_render01.jpg
[current_v2 06/48] IMG_2020_size6_render02.jpg
[current_v2 07/48] IMG_2021_size6_render00.jpg
[current_v2 08/48] IMG_2021_size6_render01.jpg
[current_v2 09/48] IMG_2021_size6_render02.jpg
[current_v2 10/48] IMG_2022_size16_render00.jpg
[current_v2 11/48] IMG_2022_size16_render01.jpg
[current_v2 12/48] IMG_2022_size16_render02.jpg
[current_v2 13/48] IMG_2023_size16_render00.jpg
[current_v2 14/48] IMG_2023_size16_render01.jpg
[current_v2 15/48] IMG_2023_size16_render02.jpg
[current_v2 16/48] IMG_2024_size16_render00.jpg
[current_v2 17/48] IMG_2024_size16_render01.jpg
[current_v2 18/48] IMG_2024_size16_render02.jpg
[current_v2 19/48] IMG_2025_size16_render00.jpg
[current_v2 20/48] IMG_2025_size16_render01.jpg
[current_v2 21/48] IMG_2025_size16_render02.jpg
[

In [7]:
# Cell 12 — Accept/reject candidate and create a non-destructive staged bundle
import shutil

def summarize(table):
    return {
        "images": len(table),
        "capacity_accuracy": float(table.capacity_correct.mean()),
        "corner_error": float(table.corner_error.mean()),
        "flavor_accuracy": float(table.flavor_correct.sum()/table.expected_occupied.sum()),
        "false_empty": int(table.false_empty.sum()),
        "count_error": int(table.count_error.sum()),
        "exact_count_accuracy": float(table.exact_counts.mean()),
    }

comparison = {}
for scope, selector in {
    "all": lambda df: df,
    "large_validation": lambda df: df[(df.split == "validation") & df.capacity.isin([30, 50])],
    "capacity_30": lambda df: df[df.capacity == 30],
    "capacity_50": lambda df: df[df.capacity == 50],
}.items():
    comparison[scope] = {
        "current_v2": summarize(selector(current_results)),
        "candidate_v3": summarize(selector(candidate_results)),
    }

current_all, candidate_all = comparison["all"]["current_v2"], comparison["all"]["candidate_v3"]
current_validation = comparison["large_validation"]["current_v2"]
candidate_validation = comparison["large_validation"]["candidate_v3"]

gates = {
    "validation_corner_improved": candidate_validation["corner_error"] < current_validation["corner_error"],
    "overall_corner_improved": candidate_all["corner_error"] < current_all["corner_error"],
    "flavor_accuracy_not_worse": candidate_all["flavor_accuracy"] >= current_all["flavor_accuracy"],
    "count_error_not_worse": candidate_all["count_error"] <= current_all["count_error"],
    "false_empty_not_worse": candidate_all["false_empty"] <= current_all["false_empty"],
    "capacity_preserved": candidate_all["capacity_accuracy"] == 1.0,
}
accepted = all(gates.values())

report = {"comparison": comparison, "gates": gates, "accepted": accepted}
(OUTPUT/"comparison.json").write_text(json.dumps(report, indent=2))

print(json.dumps(report, indent=2))

if accepted:
    SOURCE = ROOT/"data/experiments/box_localization_v2/deployment"
    STAGED = ROOT/"data/experiments/box_localization_v3/deployment"
    STAGED.mkdir(parents=True, exist_ok=True)

    shutil.copy2(SOURCE/"efficientnet_b0_box_size_classifier.pt", STAGED/"efficientnet_b0_box_size_classifier.pt")
    shutil.copy2(SOURCE/"efficientnet_b0_corner_regressor_small.pt", STAGED/"efficientnet_b0_corner_regressor_small.pt")
    shutil.copy2(CANDIDATE, STAGED/"efficientnet_b0_corner_regressor_large.pt")

    config = json.loads((SOURCE/"localization_config.json").read_text())
    config["version"] = "box_localization_v3_augmented_large"
    config["validation"] = comparison
    config["corner_routing"]["30"] = "efficientnet_b0_corner_regressor_large.pt"
    config["corner_routing"]["50"] = "efficientnet_b0_corner_regressor_large.pt"
    (STAGED/"localization_config.json").write_text(json.dumps(config, indent=2))

    print("\nPASS: candidate accepted and staged at:", STAGED)
    print("The active v2 deployment was not overwritten.")
else:
    print("\nREJECT: one or more end-to-end gates failed.")
    print("The active v2 deployment remains unchanged.")

{
  "comparison": {
    "all": {
      "current_v2": {
        "images": 48,
        "capacity_accuracy": 1.0,
        "corner_error": 0.035377602760123195,
        "flavor_accuracy": 0.7304625199362041,
        "false_empty": 3,
        "count_error": 283,
        "exact_count_accuracy": 0.4375
      },
      "candidate_v3": {
        "images": 48,
        "capacity_accuracy": 1.0,
        "corner_error": 0.023787583339781985,
        "flavor_accuracy": 0.8708133971291866,
        "false_empty": 1,
        "count_error": 129,
        "exact_count_accuracy": 0.6666666666666666
      }
    },
    "large_validation": {
      "current_v2": {
        "images": 6,
        "capacity_accuracy": 1.0,
        "corner_error": 0.06228604594235698,
        "flavor_accuracy": 0.45,
        "false_empty": 1,
        "count_error": 73,
        "exact_count_accuracy": 0.0
      },
      "candidate_v3": {
        "images": 6,
        "capacity_accuracy": 1.0,
        "corner_error": 0.04268591593182275

In [8]:
# Cell 13 — Strict held-out decision separately for 30-slot and 50-slot boxes
def summarize_strict(table):
    return {
        "images": len(table),
        "corner_error": float(table.corner_error.mean()),
        "flavor_accuracy": float(table.flavor_correct.sum()/table.expected_occupied.sum()),
        "false_empty": int(table.false_empty.sum()),
        "count_error": int(table.count_error.sum()),
        "exact_count_accuracy": float(table.exact_counts.mean()),
    }

strict_report = {"validation_by_capacity": {}, "capacity_decisions": {}}

for capacity in (30, 50):
    current = summarize_strict(current_results[(current_results.split == "validation") & (current_results.capacity == capacity)])
    candidate = summarize_strict(candidate_results[(candidate_results.split == "validation") & (candidate_results.capacity == capacity)])

    gates = {
        "corner_improved": candidate["corner_error"] < current["corner_error"],
        "flavor_not_worse": candidate["flavor_accuracy"] >= current["flavor_accuracy"],
        "count_error_not_worse": candidate["count_error"] <= current["count_error"],
        "false_empty_not_worse": candidate["false_empty"] <= current["false_empty"],
        "exact_counts_not_worse": candidate["exact_count_accuracy"] >= current["exact_count_accuracy"],
    }

    strict_report["validation_by_capacity"][str(capacity)] = {
        "current_v2": current,
        "candidate_v3": candidate,
        "gates": gates,
    }
    strict_report["capacity_decisions"][str(capacity)] = "candidate_v3" if all(gates.values()) else "current_v2"

strict_report["fully_accepted"] = all(value == "candidate_v3" for value in strict_report["capacity_decisions"].values())
(OUTPUT/"strict_heldout_decision.json").write_text(json.dumps(strict_report, indent=2))

print(json.dumps(strict_report, indent=2))
print("\nNo deployment files were changed.")

{
  "validation_by_capacity": {
    "30": {
      "current_v2": {
        "images": 3,
        "corner_error": 0.07995389302571615,
        "flavor_accuracy": 0.34444444444444444,
        "false_empty": 1,
        "count_error": 43,
        "exact_count_accuracy": 0.0
      },
      "candidate_v3": {
        "images": 3,
        "corner_error": 0.055553371691829945,
        "flavor_accuracy": 0.2777777777777778,
        "false_empty": 1,
        "count_error": 47,
        "exact_count_accuracy": 0.0
      },
      "gates": {
        "corner_improved": true,
        "flavor_not_worse": false,
        "count_error_not_worse": false,
        "false_empty_not_worse": true,
        "exact_counts_not_worse": true
      }
    },
    "50": {
      "current_v2": {
        "images": 3,
        "corner_error": 0.04461819885899781,
        "flavor_accuracy": 0.5133333333333333,
        "false_empty": 0,
        "count_error": 30,
        "exact_count_accuracy": 0.0
      },
      "candidate_v3": {